# 09 — Modelo de texto (título y garantía)

`08_errores.ipynb` mostró que más de la mitad de los errores son de vendedores con 1 sola publicación, donde el modelo depende del **contenido** de la publicación. Hoy el título entra como TF-IDF de 500 palabras sueltas. Acá pruebo sacarle más jugo al texto:

- **T1** — un modelo de texto solo (regresión logística sobre un TF-IDF grande del título), como referencia.
- **T2** — la probabilidad de `new` de ese modelo como **una feature más** del HistGB (con *cross-fitting*).
- **T3** — sumar el texto de `warranty` al modelo de texto.
- **T4** — features simples del título (largo, mayúsculas, dígitos, códigos de modelo).

Modelo fijo (`HistGradientBoostingClassifier`) y misma CV que `07`/`08`. Bloques acumulativos, una cosa por vez.

**Qué no está en el material:** TF-IDF (ya usado desde `03_titulo.ipynb`) y los n-gramas de palabras/caracteres (se explican abajo). La regresión logística y su regularización `C` sí: `material/Regresion Logistica/Presentación_Logística.pdf` y `material/Regresión Lineal Simple y Multiple/Continuación de regresion y Regulización L1 y L2.pdf`.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder, PrediccionOOF

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

resultados_bloques = []


def evaluar(nombre, pipeline):
    """Mide cualquier pipeline con la CV fija y guarda accuracy/roc_auc en resultados_bloques."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))
    return fila


def armar_pipeline_hgb(transformers_extra=(), usar_tfidf_500=True, columnas_numericas_extra=()):
    """Pipeline de la Base (capeo -> logs -> encoding -> HistGB fijo) con la opción de
    sumar transformers al ColumnTransformer o sacar el TF-IDF de 500."""
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas_base + list(columnas_numericas_extra)),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
    ]
    if usar_tfidf_500:
        transformers.append(("tfidf", TfidfVectorizer(max_features=500), "title"))
    transformers.extend(transformers_extra)

    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", ColumnTransformer(transformers, sparse_threshold=0)),
        ("modelo", modelo_fijo),
    ])

## Base

La misma de `08_errores.ipynb` (bloque K de `07`). Tiene que dar 0,9233. Si no, paro.

In [2]:
evaluar("Base", armar_pipeline_hgb())

Base -> accuracy: 0.9233 +/- 0.001 | roc_auc: 0.9783


{'bloque': 'Base',
 'accuracy_media': np.float64(0.9232857142857143),
 'accuracy_std': np.float64(0.0010010198880774112),
 'roc_auc_media': np.float64(0.9782892855376417)}

## T1 — Modelo de texto solo (referencia)

**Qué cambia respecto del TF-IDF de 500 de la Base:**

| | Base | T1 |
|---|---|---|
| Unidades | palabras sueltas | palabras sueltas **y pares de palabras** (`ngram_range=(1, 2)`) + **pedazos de 3 a 5 letras** (`analyzer="char_wb"`, `ngram_range=(3, 5)`) |
| Vocabulario | las 500 más frecuentes | todo lo que aparezca en ≥ 2 títulos (`min_df=2`) — decenas de miles de columnas |
| Modelo | HistGB (necesita matriz densa → por eso solo 500) | regresión logística, que trabaja directo sobre la matriz *sparse* gigante |

Ejemplo con el título `"Casi Nueva"`:
- palabras (1, 2): `casi`, `nueva`, `casi nueva` ← el par captura algo que las palabras sueltas no ("nueva" sola empuja a `new`, "casi nueva" es `used`).
- caracteres `char_wb` 3–5 (agrega un espacio en los bordes de cada palabra): ` ca`, `cas`, `asi`, `si `, ` cas`, … , ` nu`, `nue`, `uev`, `eva`, … ← sirven para errores de tipeo y variantes (`nuevo`/`nueva`/`nuevos`/`nuev0` comparten `nue`, `uev`).
- `strip_accents="unicode"`: `garantía` = `garantia`. `min_df=2`: descarta n-gramas que aparecen en un solo título (no generalizan).

Con tantas columnas la regresión logística puede sobreajustar, así que comparo algunos valores de `C` (inverso de la fuerza de regularización L2: `C` chico = más regularización). Es un modelo de **referencia**, no compite con el HistGB: me interesa saber cuánto sabe el texto solo, y elegir el `C` que voy a usar en T2.

In [3]:
def armar_modelo_texto(C):
    vectores = ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
    ])
    return Pipeline([
        ("vectores", vectores),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


for C in [0.5, 1, 2, 4, 8]:
    evaluar(f"T1 - texto solo (C={C})", armar_modelo_texto(C))

T1 - texto solo (C=0.5) -> accuracy: 0.8182 +/- 0.0015 | roc_auc: 0.8941


T1 - texto solo (C=1) -> accuracy: 0.8226 +/- 0.0019 | roc_auc: 0.8979


T1 - texto solo (C=2) -> accuracy: 0.8227 +/- 0.0024 | roc_auc: 0.8992


T1 - texto solo (C=4) -> accuracy: 0.8193 +/- 0.0022 | roc_auc: 0.8975


T1 - texto solo (C=8) -> accuracy: 0.8155 +/- 0.0024 | roc_auc: 0.8942


**Resultado T1.** El texto solo llega a **0,8227 de accuracy / 0,8992 de roc_auc** con `C=2`. `C=1` da prácticamente lo mismo en accuracy (0,8226), pero `C=2` tiene mejor roc_auc. Para T2 importa la **calidad de la probabilidad** (el HistGB la va a usar como número, no como clase), así que me quedo con **`C=2`**. Con `C` más chico (0,5) sobra regularización y con más grande (4, 8) empieza a sobreajustar. Es la forma de U esperable.

Como referencia: el título solo, con una regresión logística, ya clasifica bien 82 de cada 100 publicaciones, casi lo mismo que la baseline con 5 variables estructuradas (0,822). Que solo esté por debajo de la Base (0,9233) es lo esperable: no ve ni vendedor, ni categoría, ni tipo de publicación. La pregunta de T2 es si esta probabilidad le agrega al HistGB algo que el TF-IDF de 500 no le daba.

## T2 — Probabilidad del modelo de texto como feature

Envuelvo el modelo de texto de T1 (`C=2`) en `PrediccionOOF` (en `features.py`) y lo sumo al `ColumnTransformer` de la Base: el HistGB recibe **una columna más**, `prob_texto_new`.

- En el `fit` de cada fold, `PrediccionOOF.fit_transform` arma esa columna con una **CV interna de 5** (cada fila predicha por una regresión logística que no la vio) y además entrena una regresión logística con todo el train del fold.
- Al predecir la validación usa `transform`, o sea, esa regresión logística entrenada con todo el train del fold.

Por qué no es *leakage*: la validación nunca entra al ajuste (todo está adentro del `Pipeline`), y la CV interna hace que en train la feature tenga la misma calidad que va a tener en test. Si no, el HistGB aprendería a confiar en probabilidades casi perfectas (la regresión logística memoriza títulos que ya vio). Es la idea del **Stacking** (modelo de segundo nivel entrenado sobre predicciones *out-of-fold*), con la diferencia de que el segundo nivel también recibe el resto de las features.

Dos variantes:
- **T2a**: Base + `prob_texto_new` (mantiene el TF-IDF de 500).
- **T2b**: Base + `prob_texto_new` **sin** el TF-IDF de 500. ¿La probabilidad ya resume todo lo que el HistGB sacaba de esas 500 columnas?

In [4]:
modelo_texto = armar_modelo_texto(C=2)

evaluar("T2a - + prob_texto (con TF-IDF 500)",
        armar_pipeline_hgb([("texto", PrediccionOOF(modelo_texto, random_state=SEED), ["title"])]))

T2a - + prob_texto (con TF-IDF 500) -> accuracy: 0.9291 +/- 0.0017 | roc_auc: 0.9807


{'bloque': 'T2a - + prob_texto (con TF-IDF 500)',
 'accuracy_media': np.float64(0.9291142857142857),
 'accuracy_std': np.float64(0.001656526979150398),
 'roc_auc_media': np.float64(0.9807243855916029)}

In [5]:
evaluar("T2b - + prob_texto (sin TF-IDF 500)",
        armar_pipeline_hgb([("texto", PrediccionOOF(modelo_texto, random_state=SEED), ["title"])],
                           usar_tfidf_500=False))

T2b - + prob_texto (sin TF-IDF 500) -> accuracy: 0.9277 +/- 0.0016 | roc_auc: 0.98


{'bloque': 'T2b - + prob_texto (sin TF-IDF 500)',
 'accuracy_media': np.float64(0.9276714285714286),
 'accuracy_std': np.float64(0.0015808806638925713),
 'roc_auc_media': np.float64(0.980043931729966)}

**Resultado T2.**

- **T2a** (con TF-IDF 500): **0,9291 ± 0,0017**, +0,0058 sobre la Base, unas 3,4 veces el desvío entre folds. Es una ganancia real. roc_auc 0,9783 → 0,9807.
- **T2b** (sin TF-IDF 500): 0,9277 ± 0,0016. También mejora la Base (+0,0044), pero queda 0,0014 por debajo de T2a.

Esa diferencia es menor que el desvío, así que las dos son prácticamente equivalentes: la probabilidad resume **casi todo** lo que daban las 500 columnas. Igual, T2a gana en las dos métricas y el criterio es la CV, así que **sigo con T2a**. T2b queda anotada como alternativa más liviana (500 columnas menos, entrena más rápido).

## T3 — Sumar el texto de `warranty` al modelo de texto

En `08_errores.ipynb` el texto de la garantía separaba fuerte ("fábrica / oficial" → 98% `new`, "reputación / personal" → 32%), y hoy el modelo solo ve si es nula (`tiene_garantia`). La sumo al modelo de texto con **su propio** TF-IDF de palabras y pares de palabras, separado del título, para que "nuevo" en el título y "nuevo" en la garantía sean columnas distintas.

- `warranty` es nula en el 61% de las filas (en train y en test). `TfidfVectorizer` no acepta nulos, así que la paso a texto vacío `""`. Una fila vacía queda con todas sus columnas de garantía en 0. Es una transformación determinística (no aprende nada), por eso va afuera del `Pipeline`, igual que `aplanar_anidadas`.
- Solo palabras (sin n-gramas de caracteres): los textos son cortos y repetitivos ("Sí", "Sin garantía", "6 meses", "MI REPUTACION"), las palabras alcanzan.

Mido dos cosas:
1. **T3-ref**: el modelo de texto solo, con título + garantía, contra T1 (0,8227).
2. **T3**: T2a reemplazando `prob_texto_new` por la del modelo con garantía (no agrego una segunda probabilidad, la reemplazo).

In [6]:
train["warranty_texto"] = train["warranty"].fillna("")


def armar_modelo_texto_con_garantia(C):
    vectores = ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
        ("garantia", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "warranty_texto"),
    ])
    return Pipeline([
        ("vectores", vectores),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


modelo_texto_garantia = armar_modelo_texto_con_garantia(C=2)

evaluar("T3-ref - texto solo título + garantía (C=2)", modelo_texto_garantia)

T3-ref - texto solo título + garantía (C=2) -> accuracy: 0.8414 +/- 0.0019 | roc_auc: 0.918


{'bloque': 'T3-ref - texto solo título + garantía (C=2)',
 'accuracy_media': np.float64(0.8414285714285714),
 'accuracy_std': np.float64(0.0018827899402704764),
 'roc_auc_media': np.float64(0.9179925288276655)}

In [7]:
evaluar("T3 - prob_texto con garantía (con TF-IDF 500)",
        armar_pipeline_hgb([("texto", PrediccionOOF(modelo_texto_garantia, random_state=SEED),
                             ["title", "warranty_texto"])]))

T3 - prob_texto con garantía (con TF-IDF 500) -> accuracy: 0.9302 +/- 0.0012 | roc_auc: 0.9813


{'bloque': 'T3 - prob_texto con garantía (con TF-IDF 500)',
 'accuracy_media': np.float64(0.9301714285714286),
 'accuracy_std': np.float64(0.0012067835475395815),
 'roc_auc_media': np.float64(0.981279804081779)}

**Resultado T3.**

- **Solo** (T3-ref), la garantía mejora mucho al modelo de texto: 0,8227 → **0,8414** (+0,019, unas 8 veces el desvío).
- Adentro del HistGB, T3 da **0,9302 ± 0,0012**: +0,0011 sobre T2a, con roc_auc 0,9807 → 0,9813. Esa ganancia es **menor que el desvío entre folds**, o sea, está al nivel del ruido.

Es lo que anticipaba `08_errores.ipynb`: el HistGB ya tenía buena parte de esa señal por otro lado (`tiene_garantia`, tienda oficial, vendedor), y las publicaciones más difíciles (61%) no tienen garantía, así que ahí el texto no puede ayudar.

Con el mismo criterio que en `07_nuevas_features.ipynb` (fechas y precio, ganancias del orden del desvío → no se sumaron), **T3 no se acumula**: T4 se mide sobre T2a. Queda anotada como posible mejora marginal.

## T4 — Features simples del título

Cinco columnas calculadas directo del texto. Son determinísticas (no aprenden nada de los datos), así que se agregan a `train` afuera del `Pipeline`, igual que las fechas en `07`:

- `titulo_largo`: cantidad de caracteres.
- `titulo_n_palabras`: cantidad de palabras (separadas por espacios).
- `titulo_pct_mayusculas`: mayúsculas / letras. Un título "GRITADO" puede ser de un vendedor particular. Ojo: MercadoLibre suele poner la primera letra de cada palabra en mayúscula ("Sillon Antiguo Luis Xv"), así que esto mide sobre todo palabras escritas enteras en mayúscula.
- `titulo_tiene_digitos`: si aparece algún número.
- `titulo_tiene_codigo`: si hay una "palabra" que mezcla letras y números (`Gsr205`, `T20`, `Ps3`, `4691`-con-letras), típica de un código de modelo. La expresión regular `\b(?=\w*[A-Za-z])(?=\w*\d)\w+\b` pide que la palabra tenga al menos una letra **y** al menos un dígito.

Sumo las 5 juntas como un bloque sobre T2a.

In [8]:
titulo = train["title"]
cantidad_letras = titulo.str.count(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]")
cantidad_mayusculas = titulo.str.count(r"[A-ZÁÉÍÓÚÜÑ]")

train["titulo_largo"] = titulo.str.len()
train["titulo_n_palabras"] = titulo.str.split().str.len()
train["titulo_pct_mayusculas"] = (cantidad_mayusculas / cantidad_letras.replace(0, np.nan)).fillna(0)
train["titulo_tiene_digitos"] = titulo.str.contains(r"\d")
train["titulo_tiene_codigo"] = titulo.str.contains(r"\b(?=\w*[A-Za-z])(?=\w*\d)\w+\b")

columnas_titulo_simples = ["titulo_largo", "titulo_n_palabras", "titulo_pct_mayusculas",
                           "titulo_tiene_digitos", "titulo_tiene_codigo"]

train[columnas_titulo_simples].groupby(y).mean().rename(index={0: "used", 1: "new"}).round(3)

,titulo_largo,titulo_n_palabras,titulo_pct_mayusculas,titulo_tiene_digitos,titulo_tiene_codigo
condition,,,,,
used,43.384,7.102,0.183,0.407,0.105
new,47.106,7.606,0.180,0.579,0.265


In [9]:
evaluar("T4 - T2a + features simples del título",
        armar_pipeline_hgb([("texto", PrediccionOOF(modelo_texto, random_state=SEED), ["title"])],
                           columnas_numericas_extra=columnas_titulo_simples))

T4 - T2a + features simples del título -> accuracy: 0.9294 +/- 0.0015 | roc_auc: 0.9807


{'bloque': 'T4 - T2a + features simples del título',
 'accuracy_media': np.float64(0.9294),
 'accuracy_std': np.float64(0.001538353214396778),
 'roc_auc_media': np.float64(0.98071973690782)}

## Tabla final y conclusión

In [10]:
tabla_texto = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_texto["ganancia_vs_base"] = tabla_texto["accuracy_media"] - tabla_texto.loc["Base", "accuracy_media"]
tabla_texto.round(4)

,accuracy_media,accuracy_std,roc_auc_media,ganancia_vs_base
bloque,,,,
Base,0.9233,0.0010,0.9783,0.0000
T1 - texto solo (C=0.5),0.8182,0.0015,0.8941,-0.1051
T1 - texto solo (C=1),0.8226,0.0019,0.8979,-0.1007
T1 - texto solo (C=2),0.8227,0.0024,0.8992,-0.1006
T1 - texto solo (C=4),0.8193,0.0022,0.8975,-0.1040
T1 - texto solo (C=8),0.8155,0.0024,0.8942,-0.1077
T2a - + prob_texto (con TF-IDF 500),0.9291,0.0017,0.9807,0.0058
T2b - + prob_texto (sin TF-IDF 500),0.9277,0.0016,0.9800,0.0044
T3-ref - texto solo título + garantía (C=2),0.8414,0.0019,0.9180,-0.0819


**Resultado T4.** Las features simples del título **sí** difieren entre clases: las `new` tienen más dígitos (58% vs 41%) y más códigos de modelo (27% vs 11%). Pero sumadas a T2a dan 0,9294 ± 0,0015, apenas **+0,0003**, con roc_auc igual. Es ruido puro: esa información ya la tiene el modelo de texto. Los n-gramas de caracteres ven los dígitos y los códigos ("ps3", "t20"), y el largo del título se correlaciona con cuántas palabras activa el TF-IDF. Es el mismo patrón de `07`: correlación fuerte, aporte marginal casi nulo. **No se acumula.**

**Conclusión del Paso 2.**

| Bloque | accuracy | ganancia vs Base | veredicto |
|---|---|---|---|
| Base | 0,9233 ± 0,0010 | — | — |
| **T2a: + `prob_texto_new` (con TF-IDF 500)** | **0,9291 ± 0,0017** | **+0,0058** | **se suma** (≈ 3–6 veces el desvío) |
| T2b: + `prob_texto_new` (sin TF-IDF 500) | 0,9277 ± 0,0016 | +0,0044 | equivalente a T2a, más liviana |
| T3: modelo de texto con `warranty` | 0,9302 ± 0,0012 | +0,0069 (+0,0011 sobre T2a) | al límite del ruido, no se suma |
| T4: features simples del título | 0,9294 ± 0,0015 | +0,0061 (+0,0003 sobre T2a) | ruido, no se suma |

- **Lo que funcionó**: usar **todo** el vocabulario del título (pares de palabras + pedazos de letras) con una regresión logística, y pasarle al HistGB solo su probabilidad *out-of-fold*. El HistGB no puede manejar decenas de miles de columnas, pero sí una columna que las resume.
- **Lo que no**: la garantía y las features simples del título mejoran el modelo de texto solo, pero el HistGB ya tenía esa señal por otras features.
- **Nueva mejor configuración: T2a = 0,9291 / 0,9807.** Es la "Base" de los pasos que siguen.
- **Posible mejora futura marginal**: T3 (+0,0011). Si en el Paso 6 (tuning) sobra tiempo, se puede volver a medir sobre la configuración final.
